# trackslim demo

This notebook walks through the public API of **trackslim**, a small library that simplifies GPS tracks while keeping turns and elevation changes.

The main pieces are:

- `Simplifier`: the core class (Turn-Aware Elevation-Weighted Douglas-Peucker, TAEW-DP)
- `simplify_track`: a one-call convenience wrapper
- `read_gpx` / `write_gpx`: GPX input and output
- `compression_ratio`: how much a track shrank

We build a small synthetic track here so the notebook runs anywhere without extra data files.

In [ ]:
import trackslim

print("trackslim version:", trackslim.__version__)
print("Public API:", ", ".join(sorted(trackslim.__all__ if hasattr(trackslim, "__all__") else ["read_gpx", "write_gpx", "simplify_track", "Simplifier", "Track", "TrackPoint", "compression_ratio"])))

## 1. Build a track

The synthetic track has three parts: a gentle curve heading north, a sharp 90 degree turn halfway along, and a steady climb of about 40 m. A little jitter is added so that the straight parts are not perfectly collinear.

In [ ]:
import math
import tempfile
from pathlib import Path

from trackslim import Track, TrackPoint


def make_demo_track(n=400):
    """Return a deterministic synthetic Track with a turn and a climb."""
    points = []
    lat, lon = 37.5000, 127.0000
    for i in range(n):
        t = i / (n - 1)
        heading = 0.0 if t < 0.5 else 90.0  # sharp turn at the midpoint
        step = 0.00005                      # roughly 5.5 m per point
        lat += step * math.cos(math.radians(heading))
        lon += step * math.sin(math.radians(heading)) / math.cos(math.radians(lat))
        # small deterministic jitter so the track is not perfectly straight
        jitter_lat = 0.000003 * math.sin(i * 1.7)
        jitter_lon = 0.000003 * math.cos(i * 2.3)
        ele = 50.0 + 40.0 * t + 1.5 * math.sin(i / 6.0)
        points.append(TrackPoint(lat + jitter_lat, lon + jitter_lon, round(ele, 2), None))
    return Track(points=points, name="demo-ridge")


track = make_demo_track()
print("Track name:", track.name)
print("Original points:", len(track.points))
print("First point:", track.points[0])

## 2. Simplify with the `Simplifier` class

`Simplifier` takes four hyperparameters. The defaults are `tolerance_m=5.0`, `turn_weight=2.0`, `elev_weight=0.5` and `min_turn_deg=35.0`.

Points whose turn angle is at least `min_turn_deg` are always kept, so sharp corners survive even when the tolerance is large.

In [ ]:
simplifier = trackslim.Simplifier(
    tolerance_m=5.0,
    turn_weight=2.0,
    elev_weight=0.5,
    min_turn_deg=35.0,
)

simplified = simplifier.simplify(track)

print("Original points:  ", len(track.points))
print("Simplified points:", len(simplified.points))
print("Compression ratio:", round(trackslim.compression_ratio(track, simplified), 3))

## 3. Check that the turn survived

The sharp turn at the midpoint should still be represented in the simplified track. We look for a kept point near the middle of the original track and print its coordinates.

In [ ]:
mid_lat = track.points[len(track.points) // 2].lat
mid_lon = track.points[len(track.points) // 2].lon

nearest = min(
    simplified.points,
    key=lambda p: abs(p.lat - mid_lat) + abs(p.lon - mid_lon),
)

print("Turn midpoint (original):", round(mid_lat, 6), round(mid_lon, 6))
print("Nearest kept point:     ", round(nearest.lat, 6), round(nearest.lon, 6), "ele =", nearest.ele)

## 4. The one-call wrapper

For quick scripts, `simplify_track` builds a `Simplifier` for you. It takes the same tolerance and weight arguments as keyword parameters.

In [ ]:
quick = trackslim.simplify_track(track, tolerance_m=8.0, turn_weight=2.0, elev_weight=0.5)

print("Points after simplify_track:", len(quick.points))
print("Compression ratio:", round(trackslim.compression_ratio(track, quick), 3))

## 5. Effect of the tolerance

A larger tolerance removes more points. The loop below shows how the point count and compression ratio change for a few values.

In [ ]:
print(f"{'tolerance_m':>12} {'points':>8} {'ratio':>8}")
for tol in [2.0, 5.0, 10.0, 20.0]:
    result = trackslim.simplify_track(track, tolerance_m=tol)
    ratio = trackslim.compression_ratio(track, result)
    print(f"{tol:>12.1f} {len(result.points):>8d} {ratio:>8.3f}")

## 6. Write and read GPX

`write_gpx` emits a GPX 1.1 file with `<trk>`, `<trkseg>` and `<trkpt>` elements. `read_gpx` parses it back into a `Track`. We use a temporary folder so nothing is left behind in the repository.

In [ ]:
workdir = Path(tempfile.mkdtemp(prefix="trackslim_demo_"))
src_path = workdir / "demo_original.gpx"
out_path = workdir / "demo_simplified.gpx"

trackslim.write_gpx(track, src_path)
reloaded = trackslim.read_gpx(src_path)

print("Written to:", src_path)
print("Points read back:", len(reloaded.points), "of", len(track.points))

result = trackslim.simplify_track(reloaded, tolerance_m=5.0)
trackslim.write_gpx(result, out_path)
print("Simplified GPX written with", len(result.points), "points")

## 7. Inspect the simplified GPX

As a final check we read the simplified file again and compare its elevation range with the original. Keeping elevation changes is the main reason this library exists, so the first and last elevations should match closely.

In [ ]:
check = trackslim.read_gpx(out_path)

print("Simplified file points:", len(check.points))
print("Elevation range original:  ", min(p.ele for p in track.points), "to", max(p.ele for p in track.points))
print("Elevation range simplified:", min(p.ele for p in check.points), "to", max(p.ele for p in check.points))
print("Ends match:", check.points[0].ele == track.points[0].ele and check.points[-1].ele == track.points[-1].ele)

## Next steps

- Tune `turn_weight` and `elev_weight` if your tracks have very different terrain.
- Use the `elevation_guard` plugin from the command line when total ascent accuracy matters, for example on hiking tracks.
- Run `trackslim stats` on a file to see its point count and ascent before choosing a tolerance.